### import the dependencies

In [ ]:
import pandas as pd
import openai

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

## read sample dataset with a amazon inventory 

In [ ]:
df_items = pd.read_json("../../data/meta_electronics_2022_2023_with_category_ratings_100_samples_1000.jsonl", lines=True)

In [ ]:
df_items.head()

In [ ]:
list(df_items["features"].items())[0]

In [ ]:
list(df_items["images"].items())[0]

### Prprocess title and feature

In [ ]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [ ]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [ ]:
df_items["preprocessed_description"] = df_items.apply(preprocess_description, axis=1)
df_items["images"] = df_items.apply(extract_first_large_image, axis=1)

In [ ]:
df_items.head()

## sample 50 items from the dataset

In [ ]:
df_sample = df_items.sample(n=50, random_state=42)

In [ ]:
len(df_sample)

In [ ]:
df_data_to_embed = df_sample[["preprocessed_description", "images", "price", "average_rating", "parent_asin"]]

In [ ]:
df_data_to_embed.head()

In [ ]:
data_to_embed = df_data_to_embed.to_dict(orient="records")

In [ ]:
data_to_embed

### define the embedding function

In [ ]:
response = openai.embeddings.create(
    input="Random text",
    model="text-embedding-3-small"
)

In [ ]:
len(response.data[0].embedding)

In [ ]:
def getEmbedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [ ]:
getEmbedding("hi")

### create qdrant collection